<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **5. Introduction to LLMs**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 5.1 Introduction

A large language model (LLM) turns an input text, such as a question, an instruction or the start of a sentence, into an output text. It does this one **token** at a time: it looks at the text so far, predicts the next token, appends it, and starts again.

Earlier AI systems were usually built for one narrow task, for example translating English into Italian. LLMs learn general patterns of text from very large amounts of data, so the same model can translate, summarise, write code or answer questions. The Transformer architecture [01] made it feasible to train such models at scale, and within a few years they moved from research labs into search engines, office tools and code editors.

In this notebook you build a tiny next-token predictor by hand, look at the real next-token probabilities of the model on your Raspberry Pi, and see how decoding settings such as temperature turn those probabilities into text. Allow about **45 minutes**, plus time for the questions.

By the end, you should be able to explain how an LLM generates text one token at a time, why more context gives better predictions, and what temperature, top-k and top-p do to a probability distribution.

#### 5.1.1 Environment Setup

Section 5.2 runs on your laptop with plain Python. Sections 5.3 and 5.4 ask the model on your Pi, so connect the Pi as in [05.04_GettingStarted_with_Raspberry_Pi.ipynb](05.04_GettingStarted_with_Raspberry_Pi.ipynb).

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

### 5.2 How an LLM Generates Text

You have already met every step of the process in the first notebooks of this chapter:

1. The text is split into **tokens**, and each token is replaced by its ID ([05.01](05.01_Understanding_Tokenization.ipynb)).
2. Each ID selects one row of a large table, the token's **embedding** vector ([05.02](05.02_Understanding_Word_Embedding.ipynb)).
3. A stack of **attention** layers lets every token collect information from the tokens before it ([05.03](05.03_Understanding_Attention.ipynb)).
4. The last layer gives every token in the vocabulary a score. A softmax turns these scores into **probabilities** that add up to 1.
5. A **decoding** rule picks one token from these probabilities.
6. The chosen token is appended to the text, and everything starts again at step 1.

Steps 1 to 4 are the model. Step 5 is a choice made by whoever runs the model, and Section 5.4 is about that choice. Step 6 is why a long answer takes longer than a short one: every new token is another pass through the whole model.

#### 5.2.1 A tiny model

The smallest version of this loop needs no neural network at all. Our tiny model is a table: for each token, the tokens that may follow it, each with a weight.

> ```
> hello ──0.4──▶ world ──0.8──▶ !
>   │              │
>   │              └─0.2──▶ .
>   │
>   └─0.6──▶ there ──0.3──▶ !
>                   │
>                   └─0.7──▶ ,
>
> , ──1.0──▶ my ──0.8──▶ friend ──1.0──▶ !
>             │
>             └─0.2──▶ enemy ──1.0──▶ !
> ```

`generate` draws the next token according to the weights, appends it, and repeats until it reaches `!` or `.`, or a length limit. The same start gives different sentences, because every step is a random draw.

In [ ]:
tiny_model = {
    "hello": {"world": 0.4, "there": 0.6},
    "world": {"!": 0.8, ".": 0.2},
    "there": {"!": 0.3, ",": 0.7},
    ",": {"my": 1.0},
    "my": {"friend": 0.8, "enemy": 0.2},
    "friend": {"!": 1.0},
    "enemy": {"!": 1.0},
}


def generate(rules, start, window=1, max_tokens=6):
    """Draw one token at a time from the weights in `rules`. With window=2, the last
    two tokens decide wherever `rules` has an entry for them."""
    tokens = [start]
    while len(tokens) <= max_tokens and tokens[-1] not in ("!", "."):
        options = rules.get(tuple(tokens[-2:])) if window == 2 else None
        options = options or rules.get(tokens[-1])
        if not options:
            break
        tokens.append(random.choices(list(options), weights=list(options.values()))[0])
    return " ".join(tokens).replace(" ,", ",").replace(" !", "!").replace(" .", ".")


for _ in range(8):
    print(generate(tiny_model, "hello"))

#### 5.2.2 More context, better guesses

The next token may depend on more than the last token. A simple test is agreement in number: it is "this red cat", but "these red cats". We compare two settings:

- **window = 1:** only the last token decides
- **window = 2:** the last two tokens decide

In [ ]:
rules = {
    # one token of context
    "this": {"red": 0.5, "blue": 0.5},
    "these": {"red": 0.5, "blue": 0.5},
    "red": {"cat": 0.6, "cats": 0.4},
    "blue": {"cat": 0.6, "cats": 0.4},
    "cat": {".": 1.0},
    "cats": {".": 1.0},
    # two tokens of context
    ("this", "red"): {"cat": 0.9, "cats": 0.1},
    ("this", "blue"): {"cat": 0.9, "cats": 0.1},
    ("these", "red"): {"cat": 0.1, "cats": 0.9},
    ("these", "blue"): {"cat": 0.1, "cats": 0.9},
}

for start in ("this", "these"):
    for window in (1, 2):
        samples = [generate(rules, start, window) for _ in range(5)]
        print(f"{start}, window {window}:  " + "   ".join(samples))

With one token of context, the word after `red` cannot know whether the sentence started with `this` or `these`, so it is often wrong. With two tokens it knows. Real models work the same way, with a context of thousands of tokens and with probabilities learned from data instead of typed in by hand.

<div class="alert alert-block alert-info">

Change the `cat`/`cats` weights in `rules`, for one token and for two tokens of context, and rerun the cell. How often is the sentence correct now?
</div>

### 5.3 A Real Next Token

The model on your Pi does what our table does: it gives every token in its vocabulary a probability. The differences are size and origin. Its vocabulary has 100,352 tokens instead of a dozen, and its probabilities were learned from data. [05.06_Understanding_Training.ipynb](05.06_Understanding_Training.ipynb) shows how.

Ollama can return these probabilities:

- `raw=True` sends the text exactly as written, without the chat formatting, so the model simply continues it
- `logprobs=True, top_logprobs=20` returns the 20 most likely next tokens with their log-probabilities
- `num_predict=1` stops after one token, which is all we need

Ollama reports log-probabilities, the natural logarithm of each probability, because very small probabilities are easier to handle that way. `np.exp` turns them back into probabilities.

In [ ]:
import caspi

caspi.connect()
MODEL = "granite4.2:3b"
caspi.load(MODEL)


def next_token(prompt, n=20):
    """The n most likely next tokens after `prompt`, with their probabilities."""
    reply = caspi.ask(MODEL, prompt, raw=True, logprobs=True, top_logprobs=n,
                      num_predict=1, show=False)
    return {c["token"]: float(np.exp(c["logprob"])) for c in reply.logprobs[0]["top_logprobs"]}


def print_tokens(probs, n=10):
    for token, p in list(probs.items())[:n]:
        print(f"{token!r:>14}  {p:.3f}")
    print(f"{'all ' + str(len(probs)):>14}  {sum(probs.values()):.3f}")


print_tokens(next_token("The capital of Switzerland is"))

Note three details. Most candidates start with a space, because in this tokenizer the space belongs to the following word. `' Ber'` appears next to `' Bern'`: the same word, split into different pieces (see 05.01). And the 20 candidates together hold nearly all of the probability, so the other 100,000 tokens share almost nothing.

That changes when many continuations make sense:

In [ ]:
fridge = next_token("I opened the fridge and found a")
print_tokens(fridge)

Here no candidate is above about 10 %, and the 20 most likely tokens together hold only about half of the probability. The rest is spread thinly over thousands of tokens. Which one gets written is now largely a question of decoding.

### 5.4 Decoding: From Probabilities to Text

The model's work ends with the probabilities. Which token is actually written is decided by the decoding rule, and its parameters are settings of the run, not part of the model. A model may come with suggested defaults (`caspi.show()` lists them), but you can change them with every request. Each one reshapes the distribution before a token is drawn from it.

To see what they do, we apply them ourselves to the 20 candidates from the Pi. Ollama applies the same rules to the whole vocabulary, but the 20 most likely tokens are enough to see the effect. We treat them as if they were the whole vocabulary, so we first scale them up to add up to 1, and every function below returns a distribution that adds up to 1 again.

In [ ]:
def normalise(weights):
    total = sum(weights.values())
    return {token: w / total for token, w in weights.items()}


def temperature(probs, t):
    """p ** (1/t): below 1 the likely tokens gain, above 1 the distribution flattens."""
    return normalise({token: p ** (1 / t) for token, p in probs.items()})


def top_k(probs, k):
    """Keep only the k most likely tokens."""
    return normalise(dict(sorted(probs.items(), key=lambda item: -item[1])[:k]))


def top_p(probs, p):
    """Keep the most likely tokens until together they reach probability p."""
    kept = {}
    for token, q in sorted(probs.items(), key=lambda item: -item[1]):
        kept[token] = q
        if sum(kept.values()) >= p:
            break
    return normalise(kept)


def plot(distributions, titles):
    fig, axes = plt.subplots(1, len(distributions), figsize=(4.5 * len(distributions), 3.4), sharey=True)
    for ax, probs, title in zip(axes, distributions, titles):
        top = sorted(probs.items(), key=lambda item: -item[1])[:10]
        ax.bar([repr(token) for token, _ in top], [p for _, p in top], color="#4c78a8")
        ax.set_title(title, loc="left")
        ax.tick_params(axis="x", rotation=70)
    axes[0].set_ylabel("probability")
    plt.show()


fridge = normalise(fridge)

#### 5.4.1 Temperature

Temperature rescales the model's scores before they are turned into probabilities. Here that is the same as raising every probability to the power 1/T and normalising again.

- **T below 1:** the likely tokens become even more likely. The output becomes focused and repeatable, and for T close to 0 the top token always wins.
- **T = 1:** the distribution as the model produced it.
- **T above 1:** the differences shrink and unlikely tokens get a real chance. The output becomes more varied, and eventually erratic.

In [ ]:
plot([fridge, temperature(fridge, 0.5), temperature(fridge, 2.0)],
     ["T = 1 (the 20 candidates)", "T = 0.5", "T = 2"])

#### 5.4.2 Top-k

Top-k keeps only the k most likely tokens and draws among them. It cuts off the long tail of odd choices, but always keeps the same number of tokens, whatever the shape of the distribution.

- **small k:** a short list, stable and focused, but can feel dull
- **large k:** more variety, and a higher chance of an odd word

In [ ]:
plot([fridge, top_k(fridge, 3), top_k(fridge, 10)], ["all 20", "top_k = 3", "top_k = 10"])

#### 5.4.3 Top-p

Top-p, also called nucleus sampling [04], keeps the most likely tokens until together they reach probability p. Unlike top-k it adapts: when the model is confident the list stays short, and when it is unsure the list grows. Typical values are 0.9 to 0.95; we use 0.5 so that the effect shows on our 20 candidates.

In [ ]:
for prompt in ("The best way to travel from Zurich to Geneva is by", "I opened the fridge and found a"):
    kept = top_p(normalise(next_token(prompt)), 0.5)
    print(f"{prompt!r}\n   top_p = 0.5 keeps {len(kept)}: {list(kept)}\n")

#### 5.4.4 Repetition penalty and maximum length

Two more settings act on the text as a whole. A **repetition penalty** lowers the scores of tokens that already appear in the text, which stops loops like "Zurich Zurich Zurich". A **maximum length** (`num_predict` in Ollama) stops the generation after a fixed number of tokens, whatever the model was about to say. It keeps answers short, fast and cheap, at the risk of cutting them off:

In [ ]:
caspi.ask(MODEL, "Describe Zurich in one paragraph.", think=False, num_predict=25)

#### 5.4.5 From one token to whole answers

Everything in this section worked on a single next token. A real answer is a chain of such draws, and every setting applies again at every step. One unlikely token, drawn at a high temperature, becomes part of the text, and every token after it builds on that choice: this is how a model talks itself into a sentence it would never have started on purpose.

[05.12_Understanding_Inference_Control.ipynb](05.12_Understanding_Inference_Control.ipynb) takes these settings to whole answers on the Pi: how many different answers a temperature produces, how top-k and top-p keep them sensible, how a seed makes a run repeatable, and where an answer ends.

### 5.5 Further Reading and Literature

[01] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). **Attention Is All You Need.** Advances in Neural Information Processing Systems (NeurIPS). [Paper](https://arxiv.org/abs/1706.03762)

[02] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). **Language Models are Unsupervised Multitask Learners (GPT-2).** OpenAI. [Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

[03] Brown, T. B., Mann, B., Ryder, N., Subbiah, M., Kaplan, J., Dhariwal, P., et al. (2020). **Language Models are Few-Shot Learners (GPT-3).** Advances in Neural Information Processing Systems (NeurIPS). [Paper](https://arxiv.org/abs/2005.14165)

[04] Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). **The Curious Case of Neural Text Degeneration.** International Conference on Learning Representations (ICLR). [Paper](https://arxiv.org/abs/1904.09751). The paper that introduced top-p sampling.

[05] Touvron, H., Lavril, T., Izacard, G., Martinet, X., Lachaux, M. A., Lacroix, T., et al. (2023). **LLaMA: Open and Efficient Foundation Language Models.** arXiv. [Paper](https://arxiv.org/abs/2302.13971)

---
### 5.6 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: The core idea**

In your own words: what does an LLM do at every step of writing a text? Why does a long answer take longer than a short one?

(answer here)

**Question 02: The tiny model**

Change two weights in `tiny_model` from Section 5.2.1 and rerun the cell. Which sentences became more or less frequent, and why?

(answer here)

**Question 03: More context**

In Section 5.2.2, which window gets the agreement right more often? Report what you observed for both starts.

(answer here)

**Question 04: The probability of being right**

In the window-1 table, `red` is followed by `cat` with weight w1 and by `cats` with weight w2. Starting from `these`, what is the probability that the sentence ends correctly with window 1? And with window 2, where the weights after `("these", "red")` decide?

(answer here)

**Question 05: A real distribution**

Report the five most likely next tokens and their probabilities for `The capital of Switzerland is` and for a prompt of your own. How much probability do the 20 candidates hold together, and what does that say about the rest of the vocabulary?

(answer here)

**Question 06: Spaces and pieces**

Why do most candidates start with a space, and why does `' Ber'` appear next to `' Bern'`? Connect your answer to 05.01.

(answer here)

**Question 07: A factual task**

How would you set temperature, top-k and top-p for a factual task that should give the same answer every time? Use Sections 5.4.1 to 5.4.3 as evidence.

(answer here)

**Question 08: A creative task**

How would you set them for brainstorming or creative writing? What risk do you accept with those settings?

(answer here)

**Question 09: The biggest effect**

Which setting changed the fridge distribution most in your plots, and which least?

(answer here)

**Question 10: Top-k and top-p together**

Describe a realistic case for using top-k and top-p together. What guardrail does top-k add to top-p?

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)